# Kaggle setup for llm-convrec using Hugging Face Qwen3-4B-Instruct

This notebook runs the full llm-convrec pipeline on Kaggle with a Hugging Face model.

Recommended flow:
1. Install dependencies.
2. Configure the project to use `MODEL_PROVIDER: hf`.
3. Run the quick smoke test.
4. Run the full chatbot pipeline for either `restaurant` or `clothing`.

In [ ]:
import os
import pathlib
import subprocess
import yaml

# ====== EDIT THESE ======
GITHUB_REPO = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'
BRANCH = 'main'
PROJECT_DIR = pathlib.Path('/kaggle/working/llm-convrec')
DOMAIN = 'restaurant'  # change to 'clothing' if you want the clothing pipeline
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct'
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # optional; fill if model is private or rate-limited

# ====== GET SOURCE ======
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, GITHUB_REPO, str(PROJECT_DIR)], check=True)

# ====== INSTALL DEPENDENCIES ======
subprocess.run(['pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

# ====== PATCH CONFIG ======
config_path = PROJECT_DIR / 'system_config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['MODEL_PROVIDER'] = 'hf'
cfg['MODEL'] = MODEL_NAME
cfg['HF_TOKEN'] = HF_TOKEN

if DOMAIN == 'restaurant':
    cfg['PATH_TO_DOMAIN_CONFIGS'] = 'domain_specific/configs/restaurant_configs'
elif DOMAIN == 'clothing':
    cfg['PATH_TO_DOMAIN_CONFIGS'] = 'domain_specific/configs/clothing_configs'
else:
    raise ValueError("DOMAIN must be 'restaurant' or 'clothing'")

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['MODEL_PROVIDER'] = 'hf'
os.environ['MODEL'] = MODEL_NAME

print('Project directory:', PROJECT_DIR)
print('Domain:', DOMAIN)
print('Model:', MODEL_NAME)
print('Config updated for HF provider.')

In [ ]:
# Quick smoke test before the full pipeline
import subprocess

subprocess.run([
    'python',
    str(PROJECT_DIR / 'smoke_test_inference.py'),
    '--provider', 'hf',
    '--model', MODEL_NAME,
    '--prompt', 'Reply with one short sentence saying you are ready.'
], check=True)

In [ ]:
# Run the full chatbot pipeline
# Choose one of the following depending on DOMAIN

if DOMAIN == 'restaurant':
    cmd = ['python', str(PROJECT_DIR / 'restaurant_main.py')]
else:
    cmd = ['python', str(PROJECT_DIR / 'clothing_main.py')]

subprocess.run(cmd, check=True)